# 02 - Portfolio Construction & Benchmark Comparison

**Financial Portfolio Tracker** | Notebook 2 of 5

---

## Objective

Construct the weighted portfolio from individual asset returns, compare its performance against the S&P 500 benchmark (SPY), and simulate different rebalancing strategies.

## Portfolio Theory Context

A portfolio's return is the weighted average of its constituent asset returns:

$$r_p = \sum_{i=1}^{n} w_i \cdot r_i$$

where $w_i$ is the weight allocated to asset $i$ and $r_i$ is its return. This linear property makes portfolio return straightforward to compute, but portfolio *risk* is not simply the weighted average of individual risks -- it depends on correlations between assets, which is the foundation of diversification.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRADING_DAYS = 252
RISK_FREE_RATE = 0.045  # 4.5% annualised (current T-bill rate)

In [ ]:
# Load cleaned price data from Notebook 01
prices = pd.read_parquet(PROCESSED_DIR / "prices_clean.parquet")
print(f"Loaded {prices.shape[0]} trading days, {prices.shape[1]} tickers")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")
prices.tail(3)

## Define Portfolio Weights

The allocation is designed for a moderate-risk investor seeking diversification across asset classes, geographies, and risk factors.

In [ ]:
# Portfolio weights -- matches backend config exactly
WEIGHTS = {
    "VOO":  0.30,  # US Equity
    "VXUS": 0.20,  # International Equity
    "VWO":  0.10,  # Emerging Markets
    "BND":  0.20,  # Fixed Income
    "VNQ":  0.10,  # Real Estate
    "GLD":  0.10,  # Commodities
}

BENCHMARK = "SPY"
TICKERS = list(WEIGHTS.keys())

weights_series = pd.Series(WEIGHTS)
print(f"Total weight: {weights_series.sum():.2f}")
print(f"\nAllocation:")
for t, w in WEIGHTS.items():
    print(f"  {t}: {w:.0%}")

## Calculate Daily Returns

We compute simple (arithmetic) returns rather than log returns. Simple returns aggregate correctly across assets in a portfolio:

$$r_i^{\text{simple}} = \frac{P_t - P_{t-1}}{P_{t-1}}$$

Log returns are additive over time but *not* across assets, making them unsuitable for weighted portfolio construction.

In [ ]:
# Daily simple returns
returns = prices.pct_change().dropna(how="all")

print(f"Returns shape: {returns.shape}")
print(f"\nAnnualised return (individual assets):")
ann_ret = (1 + returns[TICKERS + [BENCHMARK]]).prod() ** (TRADING_DAYS / len(returns)) - 1
for t in TICKERS + [BENCHMARK]:
    print(f"  {t}: {ann_ret[t]:+.2%}")

## Weighted Portfolio Returns

The portfolio daily return is the dot product of the weight vector and the asset return vector:

$$r_p^t = \mathbf{w}^\top \cdot \mathbf{r}^t = \sum_{i=1}^{6} w_i \cdot r_i^t$$

In [ ]:
# Weighted portfolio daily returns
portfolio_returns = (returns[TICKERS].multiply(weights_series)).sum(axis=1)
portfolio_returns = portfolio_returns.dropna()
portfolio_returns.name = "Portfolio"

# Benchmark daily returns
benchmark_returns = returns[BENCHMARK].dropna()
benchmark_returns.name = "SPY"

print(f"Portfolio daily return stats:")
print(f"  Mean:   {portfolio_returns.mean():.5f} ({portfolio_returns.mean() * TRADING_DAYS:.2%} annualised)")
print(f"  Std:    {portfolio_returns.std():.5f} ({portfolio_returns.std() * np.sqrt(TRADING_DAYS):.2%} annualised)")
print(f"  Min:    {portfolio_returns.min():.4f}")
print(f"  Max:    {portfolio_returns.max():.4f}")

## Cumulative Returns: Portfolio vs Benchmark

The cumulative return shows the total wealth growth of a $1 investment:

$$\text{Cumulative Return}_T = \prod_{t=1}^{T}(1 + r_t) - 1$$

In [ ]:
# Cumulative returns
cum_portfolio = (1 + portfolio_returns).cumprod() - 1
cum_benchmark = (1 + benchmark_returns).cumprod() - 1

cum_df = pd.DataFrame({
    "Portfolio (Diversified)": cum_portfolio,
    "SPY (Benchmark)": cum_benchmark,
})

fig = px.line(
    cum_df * 100,
    title="Cumulative Returns: Diversified Portfolio vs S&P 500",
    labels={"value": "Cumulative Return (%)", "variable": "", "Date": ""},
    template="plotly_white",
    color_discrete_sequence=["#1f77b4", "#ff7f0e"],
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    annotations=[dict(
        text=f"Portfolio: {cum_portfolio.iloc[-1]:+.1%} | SPY: {cum_benchmark.iloc[-1]:+.1%}",
        xref="paper", yref="paper", x=0.02, y=0.95,
        showarrow=False, font=dict(size=13),
        bgcolor="white", bordercolor="gray", borderwidth=1,
    )]
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

fig.show()

## Individual Asset Contribution

To understand which assets drive portfolio returns, we compute each asset's cumulative contribution.

In [ ]:
# Individual cumulative returns
cum_assets = pd.DataFrame()
for t in TICKERS:
    cum_assets[t] = (1 + returns[t]).cumprod() - 1

fig = px.line(
    cum_assets * 100,
    title="Cumulative Returns by Asset",
    labels={"value": "Cumulative Return (%)", "variable": "Ticker", "Date": ""},
    template="plotly_white",
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

fig.show()

## Correlation Matrix

The diversification benefit comes from imperfect correlations. The portfolio variance is:

$$\sigma_p^2 = \sum_{i} \sum_{j} w_i w_j \sigma_i \sigma_j \rho_{ij}$$

When correlations $\rho_{ij} < 1$, the portfolio volatility is *less* than the weighted average of individual volatilities -- this is the free lunch of diversification.

In [ ]:
# Correlation heatmap
corr = returns[TICKERS + [BENCHMARK]].corr()

fig = px.imshow(
    corr.round(2),
    text_auto=".2f",
    title="Return Correlation Matrix",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    template="plotly_white",
)

fig.update_layout(height=500, width=600)
fig.show()

# Highlight lowest correlations (best diversifiers)
print("\nLowest pairwise correlations (best diversifiers):")
corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
corr_flat = corr_pairs.stack().sort_values()
for (a, b), v in corr_flat.head(5).items():
    print(f"  {a} / {b}: {v:.3f}")

## Rebalancing Simulation

Over time, asset weights drift from target allocations as different assets grow at different rates. We compare three strategies:

1. **Buy and hold** (no rebalancing): Weights drift freely
2. **Monthly rebalancing**: Reset weights to targets every month
3. **Annual rebalancing**: Reset weights to targets every year

Rebalancing is a form of contrarian investing -- it forces selling winners and buying losers, which captures mean-reversion benefits.

In [ ]:
def simulate_rebalanced_portfolio(returns_df, weights, frequency="M"):
    """
    Simulate a portfolio with periodic rebalancing.
    
    Args:
        returns_df: DataFrame of daily returns for portfolio assets.
        weights: dict of ticker -> target weight.
        frequency: 'M' for monthly, 'YE' for annual, None for buy-and-hold.
    
    Returns:
        Series of portfolio daily returns.
    """
    tickers = list(weights.keys())
    w = np.array([weights[t] for t in tickers])
    
    if frequency is None:
        # Buy and hold: weights drift with prices
        # Start with initial allocation, let it compound
        wealth = pd.DataFrame(index=returns_df.index, columns=tickers, dtype=float)
        wealth.iloc[0] = w  # Initial allocation (sums to 1)
        
        for i in range(1, len(returns_df)):
            wealth.iloc[i] = wealth.iloc[i - 1] * (1 + returns_df[tickers].iloc[i])
        
        portfolio_value = wealth.sum(axis=1)
        portfolio_ret = portfolio_value.pct_change().dropna()
        return portfolio_ret
    
    # Periodic rebalancing
    rebal_dates = returns_df.resample(frequency).last().index
    current_weights = w.copy()
    daily_returns = []
    
    for date in returns_df.index:
        day_ret = returns_df.loc[date, tickers].values.astype(float)
        port_ret = np.dot(current_weights, day_ret)
        daily_returns.append(port_ret)
        
        # Update weights based on asset returns
        current_weights = current_weights * (1 + day_ret)
        current_weights = current_weights / current_weights.sum()
        
        # Rebalance on rebalancing dates
        if date in rebal_dates:
            current_weights = w.copy()
    
    return pd.Series(daily_returns, index=returns_df.index, name=f"Rebal_{frequency}")


# Run simulations
ret_clean = returns[TICKERS].dropna()

buy_hold = simulate_rebalanced_portfolio(ret_clean, WEIGHTS, frequency=None)
monthly_rebal = simulate_rebalanced_portfolio(ret_clean, WEIGHTS, frequency="M")
annual_rebal = simulate_rebalanced_portfolio(ret_clean, WEIGHTS, frequency="YE")

print("Rebalancing simulations computed.")

In [ ]:
# Compare rebalancing strategies
rebal_cum = pd.DataFrame({
    "Buy & Hold": (1 + buy_hold).cumprod() - 1,
    "Monthly Rebalancing": (1 + monthly_rebal).cumprod() - 1,
    "Annual Rebalancing": (1 + annual_rebal).cumprod() - 1,
})

fig = px.line(
    rebal_cum * 100,
    title="Rebalancing Strategy Comparison",
    labels={"value": "Cumulative Return (%)", "variable": "Strategy", "Date": ""},
    template="plotly_white",
    color_discrete_sequence=["#636efa", "#00cc96", "#ef553b"],
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

fig.show()

# Summary statistics
for name, ret in [("Buy & Hold", buy_hold), ("Monthly", monthly_rebal), ("Annual", annual_rebal)]:
    ann_r = (1 + ret).prod() ** (TRADING_DAYS / len(ret)) - 1
    ann_v = ret.std() * np.sqrt(TRADING_DAYS)
    sr = (ann_r - RISK_FREE_RATE) / ann_v if ann_v > 0 else 0
    print(f"{name:20s}  Return: {ann_r:+.2%}  Vol: {ann_v:.2%}  Sharpe: {sr:.3f}")

## Weight Drift Visualization

Without rebalancing, the portfolio's allocation drifts as winners grow and losers shrink. This can lead to unintended concentration risk.

In [ ]:
# Simulate weight drift under buy-and-hold
prices_norm = prices[TICKERS].divide(prices[TICKERS].iloc[0])
weighted_values = prices_norm.multiply(weights_series)
total_value = weighted_values.sum(axis=1)
drifted_weights = weighted_values.divide(total_value, axis=0)

fig = px.area(
    drifted_weights * 100,
    title="Portfolio Weight Drift (Buy & Hold)",
    labels={"value": "Weight (%)", "variable": "Asset", "Date": ""},
    template="plotly_white",
)

fig.update_layout(
    height=450,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

fig.show()

# Show current drifted weights vs target
current_drift = drifted_weights.iloc[-1]
print("\nCurrent drifted weights vs target:")
for t in TICKERS:
    drift = current_drift[t] - WEIGHTS[t]
    print(f"  {t}: {current_drift[t]:.1%} (target {WEIGHTS[t]:.0%}, drift {drift:+.1%})")

## Tracking Error Analysis

Tracking error measures how much the portfolio's returns deviate from the benchmark:

$$\text{TE} = \sqrt{\frac{252}{N} \sum_{t=1}^{N}(r_p^t - r_b^t)^2}$$

A higher tracking error means the portfolio behaves differently from SPY -- expected for a diversified multi-asset portfolio.

In [ ]:
# Active returns (portfolio - benchmark)
active_returns = portfolio_returns - benchmark_returns
active_returns = active_returns.dropna()

# Annualised tracking error
te = active_returns.std() * np.sqrt(TRADING_DAYS)

# Rolling tracking error (60-day window)
rolling_te = active_returns.rolling(60).std() * np.sqrt(TRADING_DAYS)

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Daily Active Returns (Portfolio - SPY)", "Rolling 60-Day Tracking Error")
)

fig.add_trace(
    go.Bar(x=active_returns.index, y=active_returns * 100, name="Active Return",
           marker_color=np.where(active_returns >= 0, "#26a69a", "#ef5350")),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=rolling_te.index, y=rolling_te * 100, name="Tracking Error",
               line=dict(color="#1f77b4", width=2)),
    row=2, col=1
)

fig.update_layout(height=600, template="plotly_white", showlegend=False)
fig.update_yaxes(title_text="Active Return (%)", row=1, col=1)
fig.update_yaxes(title_text="Tracking Error (%)", row=2, col=1)

fig.show()

print(f"\nAnnualised Tracking Error: {te:.2%}")
print(f"Information Ratio: {(portfolio_returns.mean() - benchmark_returns.mean()) * TRADING_DAYS / te:.3f}")

## Save Portfolio Returns

Save computed return series for use in subsequent notebooks.

In [ ]:
# Save returns for downstream notebooks
returns_output = returns[TICKERS + [BENCHMARK]].copy()
returns_output["Portfolio"] = portfolio_returns
returns_output.to_parquet(PROCESSED_DIR / "returns.parquet")

print(f"Saved returns to {PROCESSED_DIR / 'returns.parquet'}")
print(f"Shape: {returns_output.shape}")
print(f"Columns: {list(returns_output.columns)}")

## Summary

**Key Findings:**

1. **Diversification cost**: The diversified portfolio trades off some upside from pure US equity (SPY) for lower volatility and more stable drawdowns.
2. **Correlation structure**: BND and GLD show low or negative correlation with equity assets, providing genuine diversification benefit. This aligns with their role as portfolio stabilizers during equity drawdowns.
3. **Rebalancing impact**: Monthly rebalancing tends to slightly improve risk-adjusted returns (Sharpe ratio) compared to buy-and-hold, by systematically selling high and buying low.
4. **Weight drift**: Without rebalancing, VOO's weight increases over time as US equities outperform, gradually shifting the portfolio toward higher concentration.

**Next step**: Notebook 03 -- Deep performance analysis with drawdowns, rolling windows, and calendar returns.